In [4]:
import numpy as np
import pandas as pd

In [5]:
from surprise import SVD, KNNBaseline, Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy
from surprise.model_selection import cross_validate

In [6]:
import time

In [7]:
path = "interactions_data/"

In [8]:
df_movies = pd.read_csv(path + 'movies_data.csv')
df_interactions = pd.read_csv(path + 'interactions_data.csv')

In [9]:
df_movies.rename(columns={'id': 'movie_id', 'title': 'title', 'genres': 'genres'}, inplace=True)

In [10]:
df_movies.head()

,movie_id,title,description,genres,actors,directors
0,37702,Forbidden City Cop,An imperial agent gets ridiculed for his vario...,"Adventure, Action, Comedy","Stephen Chow, Carina Lau, Carman Lee Yeuk-Tung...",Vincent Kok Tak-Chiu
1,1290821,Shelter,A man living in self-imposed exile on a remote...,"Action, Thriller, Crime","Jason Statham, Bodhi Rae Breathnach, Bill Nigh...",Ric Roman Waugh
2,1171145,Crime 101,When an elusive thief whose high-stakes heists...,"Thriller, Crime","Chris Hemsworth, Mark Ruffalo, Halle Berry, Ba...",Bart Layton
3,1198994,Send Help,Two colleagues become stranded on a deserted i...,"Horror, Comedy, Thriller","Rachel McAdams, Dylan O'Brien, Edyll Ismail, D...",Sam Raimi
4,840464,Greenland 2: Migration,Having found the safety of the Greenland bunke...,"Adventure, Thriller, Science Fiction","Gerard Butler, Morena Baccarin, Roman Griffin ...",Ric Roman Waugh


In [11]:
df_interactions.head()

,user_id,movie_id,rating,max_progress_percent,favourite,in_watchlist,view_count,last_watched_at
0,10,69492,NaN,85,0,1,3,2025-12-18 01:56:05.017054
1,10,28932,NaN,80,0,0,1,2026-02-02 01:56:05.032371
2,10,372754,NaN,85,0,1,2,2025-12-10 01:56:05.042977
3,10,210479,NaN,82,0,0,6,2025-10-20 01:56:05.053147
4,10,16071,NaN,80,0,0,7,2025-10-30 01:56:05.064748


In [12]:
def calculate_base_score(row, weights):
    v_progress = row['max_progress_percent'] / 100
    v_rating = row['rating'] / 5.0 if pd.notnull(row['rating']) else None
    v_fav = float(row['favourite'])
    v_watchlist = float(row['in_watchlist'])
    v_view_count = np.log(max(row['view_count'], 1)) / np.log(10)

    # Tính điểm theo từng thành phần hợp lệ
    numerator = (
        weights['progress'] * v_progress
        + weights['favourite'] * v_fav
        + weights['watchlist'] * v_watchlist
        + weights['view_count'] * v_view_count
    )
    denominator = (
        weights['progress']
        + weights['favourite']
        + weights['watchlist']
        + weights['view_count']
    )

    # Chỉ cộng rating khi có dữ liệu
    if v_rating is not None:
        numerator += weights['rating'] * v_rating
        denominator += weights['rating']

    return numerator / denominator * 10

In [13]:
weights = {
    'progress': 3.0,
    'rating': 4.1,
    'favourite': 4.5,
    'watchlist': 2.0,
    'view_count': 1.1
}

df_interactions['base_score'] = df_interactions.apply(calculate_base_score, axis=1, weights=weights)

In [ ]:
df_interactions

,user_id,movie_id,rating,max_progress_percent,favourite,in_watchlist,view_count,last_watched_at,base_score
0,10,69492,NaN,85,0,1,3,2025-12-18 01:56:05.017054,4.787579
1,10,28932,NaN,80,0,0,1,2026-02-02 01:56:05.032371,2.264151
2,10,372754,NaN,85,0,1,2,2025-12-10 01:56:05.042977,4.604842
3,10,210479,NaN,82,0,0,6,2025-10-20 01:56:05.053147,3.128270
4,10,16071,NaN,80,0,0,7,2025-10-30 01:56:05.064748,3.141139
...,...,...,...,...,...,...,...,...,...
27011,500,15276,5.0,97,1,0,4,2025-12-06 01:59:01.806758,8.280453
27012,500,736545,5.0,81,1,1,3,2026-03-08 01:59:01.818289,9.220975
27013,500,11297,4.0,92,0,0,2,2025-10-01 01:59:01.826666,4.334104
27014,500,146724,5.0,90,1,0,3,2025-10-04 01:59:01.838193,8.044104


: 

: 

: 

: 

In [14]:
from collections import Counter

In [ ]:
# Algorithm 1: Phân loại thể loại yêu thích và không thích của người dùng
# Lấy trung vị (median)

def get_user_genres_preferences(user_id, df_interaction, df_items):
  # Get movieId which user rated
  user_ratings = df_interactions[df_interaction['user_id'] == user_id]

  # Count genre
  genre_counts = Counter()
  for m_id in user_ratings['movie_id']:
    genre_str = df_items.loc[df_items['movie_id'] == m_id, 'genres'].values[0]
    if pd.isna(genre_str): continue
    for genre in genre_str.split(', '):
      genre_counts[genre] += 1

  # Get all genre
  all_genres = set()
  df_items['genres'].str.split(', ').apply(lambda x: all_genres.update(x) if isinstance(x, list) else None)

  # Sort asc
  sorted_genres = sorted(list(all_genres), key=lambda x: genre_counts.get(x, 0), reverse=True)

  # Find mid-point
  mid_point = len(sorted_genres) // 2
  fav_genres = sorted_genres[:mid_point]
  unfav_genres = sorted_genres[mid_point:]

  return fav_genres, unfav_genres


In [16]:
# Algorithm 2: Kiểm tra tính liên quan (Relevance) dựa trên Genre
def is_movie_relevant(movie_id, df_movies, fav_genres, unfav_genres):
    res = df_movies.loc[df_movies['movie_id'] == movie_id, 'genres'].values
    if len(res) == 0 or pd.isna(res[0]): return False

    movie_genres = [g.strip() for g in res[0].split(', ')]
    # Score = (Số genre thích) - (Số genre ghét)
    score = sum(1 for g in movie_genres if g in fav_genres) - \
            sum(1 for g in movie_genres if g in unfav_genres)
    return score > 0

In [17]:
# Algorithm 4: Tính các chỉ số Precision@K, Recall@K, F1 score
def calculate_metrics_at_k(predictions, df_interaction, df_movies, k):
  user_est_true = {}
  for uid, iid, true_r, est, _ in predictions:
    if uid not in user_est_true:
      user_est_true[uid] = []
    user_est_true[uid].append((iid, est))

  precisions = []
  recalls = []

  for uid, user_ratings in user_est_true.items():
    # Lấy gu của user
    fav, unfav= get_user_genres_preferences(uid, df_interaction, df_movies)

    # Sắp xếp dự đoán theo điểm 'est' cao nhất
    user_ratings.sort(key=lambda x: x[1], reverse=True)
    top_k = user_ratings[:k]

    # Tính TP (True Positives)
    n_rel_and_rec_k = sum(1 for (iid, est) in top_k if is_movie_relevant(iid, df_movies, fav, unfav))

    # Tính tổng số phim relevant thực tế có trong tập test của user
    n_rel = sum(1 for (iid, est) in user_ratings if is_movie_relevant(iid, df_movies, fav, unfav))

    # Tính precision and recall cho user
    precisions.append(n_rel_and_rec_k / k)
    recalls.append(n_rel_and_rec_k / n_rel if n_rel != 0 else 0)

  # Lấy avg
  avg_prec = np.mean(precisions)
  avg_rec = np.mean(recalls)

  f1 = (2 * avg_prec * avg_rec) / (avg_rec + avg_prec) if (avg_rec + avg_prec) >0 else 0

  return avg_prec, avg_rec, f1

In [18]:
from surprise import SVD, KNNBaseline, CoClustering, NMF, SlopeOne, BaselineOnly

In [19]:
# Danh sách các thuật toán
algorithms = [
    SVD(),
    CoClustering(),
    NMF(),
    SlopeOne(),
    KNNBaseline(),
    BaselineOnly()
]

In [20]:
# Chia dữ liệu Trainning và test, tỉ lệ 75/25
reader = Reader(rating_scale=(0, 10))

data = Dataset.load_from_df(df_interactions[['user_id', 'movie_id', 'base_score']], reader)

trainset, testset = train_test_split(data, test_size=0.25)

In [21]:
# Huấn luyện, dự đoán và đo thời gian
results_table = []

for algorithm in algorithms:
  algo_name = algorithm.__class__.__name__

  start_time = time.time()

  # Train
  algorithm.fit(trainset)

  # Predict
  predictions = algorithm.test(testset)

  #Time
  eval_time = time.time() - start_time


  # Tinh RMSE va MAE
  rmse_val = accuracy.rmse(predictions, verbose=False)
  mae_val = accuracy.mae(predictions, verbose=False)

  # Tinh Precision, recall, f1
  prec, rec, f1 = calculate_metrics_at_k(predictions, df_interactions, df_movies, k = 15)

  results_table.append([algo_name, rmse_val, mae_val, prec, rec, f1, eval_time])
  print(f"{algo_name:<15} | {rmse_val:<8.4f} | {mae_val:<8.4f} | {prec:<10.4f} | {rec:<8.4f} | {f1:<8.4f} | {eval_time:<8.4f}")


SVD             | 1.9135   | 1.4325   | 0.7308     | 0.9461   | 0.8246   | 0.2981  
CoClustering    | 2.5967   | 2.0447   | 0.7321     | 0.9475   | 0.8260   | 0.9175  
NMF             | 2.4806   | 1.9013   | 0.7331     | 0.9484   | 0.8270   | 0.8319  
SlopeOne        | 2.2531   | 1.6876   | 0.7320     | 0.9473   | 0.8258   | 0.9080  
Estimating biases using als...
Computing the msd similarity matrix...
Done computing similarity matrix.
KNNBaseline     | 2.1192   | 1.6024   | 0.7309     | 0.9463   | 0.8247   | 0.0645  
Estimating biases using als...
BaselineOnly    | 1.9427   | 1.4677   | 0.7306     | 0.9459   | 0.8244   | 0.0410  


In [22]:

# Xuất ra DataFrame 
df_final_results = pd.DataFrame(results_table, columns=['Algorithms', 'RMSE', 'MAE', 'Precision', 'Recall', 'F1 Score', 'Time'])
df_final_results

,Algorithms,RMSE,MAE,Precision,Recall,F1 Score,Time
0,SVD,1.913461,1.432537,0.730754,0.946126,0.824609,0.298073
1,CoClustering,2.596687,2.044670,0.732111,0.947544,0.826012,0.917503
2,NMF,2.480568,1.901252,0.733062,0.948438,0.826957,0.831924
3,SlopeOne,2.253104,1.687570,0.731976,0.947340,0.825848,0.907996
4,KNNBaseline,2.119230,1.602381,0.730889,0.946251,0.824743,0.064541
5,BaselineOnly,1.942659,1.467692,0.730618,0.945925,0.824446,0.040965
